# 使用 DeepSeek 实现 RAG 应用实践

在这节课程中，我们将一起探索如何利用强大的大型语言模型（LLM）和一种名为 RAG 的前沿技术，来构建一个能“学习”私有知识并进行问答的智能机器人。

## 课程目标

*   **核心目标:** 亲手构建一个针对自定义知识库（一篇关于《鲁米纳的星语者》的介绍）的智能问答 (Q&A) 机器人。
*   **知识目标:**
    *   理解什么是 RAG (Retrieval-Augmented Generation) 及其解决了大模型的什么痛点。
    *   识别 RAG 应用的核心组件（文档加载、文本切分、向量化、向量存储、检索、大模型生成）。
*   **能力目标:**
    *   能够使用 DeepSeek 和 LangChain 框架，完成一个端到端的 RAG 应用。
    *   能够对 RAG 应用的结果进行初步的测试和评估。

## 1. 欢迎来到 RAG 的世界

你有没有试过问 AI一些很新的问题，或者关于你自己文档的问题？比如：“帮我总结一下我上周写的会议纪要？”

通常，AI 会回答：“抱歉，我不知道。”或者“我的知识截止到202X年。”

这就是传统大模型的局限性。它们虽然博学，但知识库是静态的，并且无法访问你的私有数据。这不仅导致了 **知识盲区**，有时还会产生“一本正经地胡说八道”的 **模型幻觉**。

**那么，如何让 AI 学会我们想让它知道的知识呢？**

这就是我们今天的主角—— **RAG (Retrieval-Augmented Generation)** 发光发热的地方！


**RAG**，全称是“检索增强生成”。

我们可以用一个生动的比喻来理解它：**给大模型一本可以随时查阅的开卷考试参考书**。

当我们向 AI 提问时，它不再仅仅依赖自己“大脑”里固有的知识，而是会经历以下步骤：

1.  **检索 (Retrieval):** 首先，AI 会去我们给它的“参考书”（也就是我们的知识库）里，快速找到与问题最相关的几段内容。
2.  **增强 (Augmented):** 然后，它会将我们的原始问题和找到的相关内容 **“打包”** 在一起。
3.  **生成 (Generation):** 最后，AI 会像一个聪明的学生一样，根据打包好的问题和参考资料，生成一个精准、可靠的答案。

简单的 RAG 工作流程：

![](https://imgbed.momodel.cn/20250319143754947.png)

1. **数据收集**：收集需要检索的数据信息，如文档或网页。

2. **数据分块**：将收集的信息分割成 chunk，并编码为向量。

3. **文档嵌入**：将编码后的向量嵌入到向量数据库中。

4. **处理用户查询**：接收到用户问题，将问题转换为向量，并在向量数据库中查找与其最相关的文档块。

5. **使用 LLM 生成回答**：将检索到的文档块与问题一起作为提示（prompt） 输入到 LLM 中，生成最终回答。


通过这个流程，AI 就能回答基于我们特定知识库的问题了！是不是很酷？接下来，我们就来准备工具，亲手实现它！

## 2. 准备我们的神兵利器

工欲善其事，必先利其器。在开始编码之前，我们需要准备好几样“神兵利器”。

### 2.1 认识 DeepSeek

[DeepSeek](https://www.deepseek.com/) ，国内顶尖人工智能公司开发的平台，提供了非常强大且性价比极高的大语言模型服务。我们将使用它的模型作为我们 RAG 应用的“大脑”。

### 2.2 获取“魔法钥匙”：API Key

要使用 DeepSeek 的服务，我们需要一把“魔法钥匙”，也就是 API Key。

1.  访问 [DeepSeek 官网](https://platform.deepseek.com/sign_up)。
2.  注册一个账户。
3.  在个人账户页面找到“API 密钥”或“API Keys”选项。
4.  创建一个新的密钥，并**立即复制并保存好它**。为了安全，这个密钥只会完整显示一次。

**请注意：** API Key 是你的个人凭证，非常重要，请妥善保管，不要泄露给他人！

### 2.3 搭建开发环境

接下来，我们需要安装一些 Python 库。这些库就像是我们的工具箱，能帮助我们更方便地构建应用。

请在你的终端或命令提示符中运行以下命令来检查所有必要的库：

In [ ]:
import importlib
# 检查核心库是否都已安装
libraries = [ "langchain_deepseek", "langchain", "langchain_community", "sentence_transformers"]
all_installed = True
for lib in libraries:
    try:
        importlib.import_module(lib)
    except ImportError:
        print(f"警告：库 {lib} 未安装。请运行上面的 pip install 命令。")
        all_installed = False

if all_installed:
    print("✅所有必要的库都已成功安装！")

我们来简单认识一下这些库：
*   `langchain_deepseek`: DeepSeek 的模型套件，方便我们在 langchian 调用它的模型。
*   `langchain` & `langchain_community`: 一个强大的开源框架，能极大地简化 LLM 应用的开发，我们将用它来“编排”整个 RAG 流程。
*   `sentence-transformers`: 一个神奇的库，能轻松地将文本句子转换为具有语义信息的向量（也就是 Embedding）。

## 3. 动手搭建你的第一个 RAG 应用

理论和准备工作都已就绪，让我们卷起袖子，开始课程最激动人心的部分——编码！

### 3.1 准备工作：配置你的 API Key

为了让我们的代码能连接到 DeepSeek，我们需要先配置好 API Key。

**请注意：** 直接在代码中写入 Key 是不安全的做法。在实际项目中，我们通常使用环境变量来管理。但为了本次课程的简洁性，我们先将 Key 填入下面的代码中。

另外还需要填入你的deepseek_api_url，如果你是按照之前课件中的方式来进行获取的，就填入`https://api.deepseek.com/v1/generate`

In [ ]:
import os
from getpass import getpass

# 推荐使用 getpass 提示用户输入，避免在代码中明文暴露
# 如果你在一个无法交互的环境中运行，可以将你的 Key 直接赋值给 os.environ["DEEPSEEK_API_KEY"]
if "DEEPSEEK_API_KEY" not in os.environ:
    os.environ["DEEPSEEK_API_KEY"] = getpass("请输入你的 DeepSeek API Key: ")

if "DEEPSEEK_API_URL" not in os.environ:
    os.environ["DEEPSEEK_API_URL"] = getpass("请输入你的 DEEPSEEK API URL: ")


In [ ]:
import os
import requests

# 设置 API 请求参数
api_url = os.environ['DEEPSEEK_API_URL']
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"
}
data = {
    "model": "deepseek-r1-250528",
    "messages": [
        {"role": "system", "content": "你是人工智能助手."},
        {"role": "user", "content": "鲁米纳星球面临什么危险？"}
    ]
}

# 发送 POST 请求
response = requests.post(api_url, json=data, headers=headers)

# 检查响应
if response.status_code == 200:
    print("✅ 请求成功！")
    response_data = response.json()
    # 提取并打印可读的文本内容
    if 'choices' in response_data and len(response_data['choices']) > 0:
        message_content = response_data['choices'][0]['message']['content']
        print(message_content)
    else:
        print("未找到有效的回复内容。")
else:
    print(f"❌ 请求失败，状态码：{response.status_code}")
    print(response.text)

#### 结果分析

看到了吗？模型的回答非常“通用”和模糊。它可能会根据训练数据中关于科幻小说的普遍知识进行猜测，但它无法给出我们知识库中具体、精准的原因。它甚至可能会承认自己不知道确切细节。

这就是 RAG 需要解决的问题。接下来，我们将一步步为模型装上“参考书”，让它成为一个鲁米纳专家！

### 3.2 第一步：加载你的知识库

首先，我们需要把我们的“参考书”——`knowledge_base.txt` 文件加载进来。LangChain 提供了各种 `DocumentLoader` 来处理不同类型的文件。对于简单的 `.txt` 文件，我们可以使用 `TextLoader`。

In [ ]:
from langchain_community.document_loaders import TextLoader

# 指定我们的知识库文件名
file_path = "knowledge_base.txt"

# 创建一个加载器实例
loader = TextLoader(file_path, encoding="utf-8")

# 加载文档
documents = loader.load()

# --- 即时反馈练习 ---
# 使用断言来验证我们是否成功加载了文档
assert len(documents) == 1, "文档加载失败！"
print("✅ 文档加载成功！")

# 让我们看一看加载进来的文档内容
print(f"\n共加载了 {len(documents)} 篇文档。")
print("\n文档内容预览：")
print(documents[0].page_content[:200])  # 打印前200个字符

### 3.3 第二步：为知识“分段” (Text Splitting)

加载进来的文档通常很长，直接把整篇文档丢给模型效率不高，效果也不好。因此，我们需要把长文档切分成一个个更小的、逻辑上连贯的文本块 (Chunks)。

**为什么要切分？**
1.  **上下文窗口限制:** 大模型一次能处理的文本长度是有限的（称为“上下文窗口”）。
2.  **检索效率与精度:** 更小的文本块能让检索更集中于问题的核心，提高检索的准确性。

#### 可视化理解文本切分
`chunk_overlap` 参数非常重要，它能让相邻的文本块之间有一部分内容是重叠的，这就像书页的页眉页脚一样，有助于在切分处保持上下文的连贯性。

让我们用一个简单的图来理解这个过程：

![](https://imgbed.momodel.cn/20250710094605123.png)

#### 动手练习：交互式文本切分

现在，我们将使用 `ipywidgets` 来创建一个交互式体验。你可以拖动下面的滑块来实时调整 `chunk_size` 和 `chunk_overlap`，并观察它们是如何影响最终的切分结果的。

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import ipywidgets as widgets
from IPython.display import display

# 获取我们加载的文档内容
doc_content = documents[0].page_content


def interactive_text_splitter(chunk_size, chunk_overlap):
    """根据给定的参数切分文本并显示结果"""
    if chunk_overlap >= chunk_size:
        print("❌ 错误：重叠长度 (chunk_overlap) 不能大于或等于块长度 (chunk_size)。")
        return

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    split_docs = text_splitter.split_text(doc_content)

    print(f"切分参数: Chunk Size = {chunk_size}, Chunk Overlap = {chunk_overlap}")
    print(f"切分后的文本块数量: {len(split_docs)}")
    print("-" * 20)
    
    # 预览前两个文本块和它们之间的关系
    if not split_docs:
        print("文本内容为空或过短，无法切分。")
        return

    print("【第一块】\n", split_docs[0])
    
    if len(split_docs) > 1:
        if chunk_overlap > 0:
            # 找到并打印重叠的文本
            overlap_text = split_docs[0][-chunk_overlap:]
            print("\n" + "=" * 10 + f" (重叠部分: '{overlap_text}') " + "=" * 10 + "\n")
        else:
            print("\n" + "=" * 10 + " (无重叠) " + "=" * 10 + "\n")
        
        print("【第二块】\n", split_docs[1])

# 创建交互式滑块
widgets.interactive(
    interactive_text_splitter,
    chunk_size=widgets.IntSlider(
        min=100, max=1000, step=50, value=200, description="块长度"
    ),
    chunk_overlap=widgets.IntSlider(
        min=0, max=200, step=10, value=20, description="重叠长度"
    ),
)

**请在这里进行你的探索和实验。** 完成后，我们将使用一组固定的参数来为我们的应用进行最终的文本切分。

In [ ]:
# 确定最终的切分参数
final_chunk_size = 200
final_chunk_overlap = 20

# 使用最终参数创建文本切分器
final_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=final_chunk_size,
    chunk_overlap=final_chunk_overlap,
    length_function=len,
)

# 对文档进行最终切分
split_docs = final_text_splitter.split_documents(documents)

# --- 即时反馈练习 ---
assert len(split_docs) > 1, "文本切分似乎没有生效！"
print(f"✅ 文本成功切分为 {len(split_docs)} 个块。")

### 3.4 第三步：让文字变成“数字密码” (Embedding)

计算机不理解文字的含义，但它们擅长处理数字。`Embedding` 就是一座桥梁，它能将我们的文本块转换成一串串数字，也就是 **向量 (Vector)**。这些向量可以捕捉到文本的语义信息。

简单来说，意思相近的文本，它们的向量在空间中的距离也相近。

我们将使用 `sentence-transformers` 库中的一个优秀开源模型来完成这个任务。

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# 指定我们要使用的开源 Embedding 模型
# 'paraphrase-multilingual-MiniLM-L12-v2' 是一个轻量且高效的多语言模型
model_name = "/home/jovyan/work/datasets/68b55c157db17032a910bec9-momodel/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {'device': 'cpu'}  # 使用 CPU 进行计算
encode_kwargs = {'normalize_embeddings': True} # 设为 True 以便计算余弦相似度

# 创建一个 LangChain 兼容的 Embedding 模型实例
embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name, 
    model_kwargs=model_kwargs, 
    encode_kwargs=encode_kwargs
)

# 让我们来测试一下，将一句话转换为向量
test_text = "你好，世界！"
try:
    # 使用模型的 encode 方法生成向量 (效率更高)
    # 可以指定 show_progress_bar=True 来显示进度条
    test_vector = embeddings_model.embed_query(test_text)
except Exception as e:
    print(f"生成向量时出错: {e}")

print(f"'{test_text}' 对应的向量（部分）为：\n{test_vector[:10]}...")
print(f"\n向量的维度为：{len(test_vector)}")

# --- 即时反馈练习 ---
assert len(test_vector) == 384, "Embedding 模型的维度不正确！"
print("\n✅ Embedding 模型加载成功，向量维度正确。")

### 3.5 第四步：构建“数字密码”的智能索引 (Vector Store)

现在我们有了所有文本块的“数字密码”（向量），我们需要一个高效的系统来存储和检索它们。这个系统就是 **向量数据库 (Vector Store)**。

当用户提问时，我们会先把问题也转换成一个向量，然后去向量数据库里，找到和问题向量“距离最近”的几个文本块向量，这些文本块就是与问题最相关的内容。

我们使用 `FAISS` (Facebook AI Similarity Search) 来构建这个本地的向量数据库。

In [ ]:
!pip install faiss-gpu

In [ ]:
from langchain_community.vectorstores import FAISS

# 使用 FAISS 从切分好的文档和 Embedding 模型构建向量数据库
# 这个过程会需要一些时间，因为它正在为所有文本块计算 Embedding
print("正在构建向量数据库，请稍候...")
vector_store = FAISS.from_documents(split_docs, embeddings_model)
print("✅ 向量数据库构建完成！")

### 3.6 第五步：召唤 DeepSeek 神龙，实现问答！

万事俱备，只欠东风！现在，我们终于可以把所有组件“串”起来，召唤 DeepSeek 大模型来回答问题了。

整个流程是：
1.  **初始化 DeepSeek LLM**: 创建一个 DeepSeek 模型的实例。
2.  **创建检索器 (Retriever)**: 将我们的向量数据库设置为一个检索器，它能根据问题检索相关文档。
3.  **创建问答链 (QA Chain)**: 使用 LangChain 的 `RetrievalQA` 链，它会自动处理“检索->增强->生成”的整个过程。

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import requests
import json

# 简单的 DeepSeek 调用函数
def call_deepseek(prompt: str) -> str:
    """调用 DeepSeek API"""
    data = {
        "model": "deepseek-r1-250528",
        "messages": [
            {"role": "system", "content": "你是一个有帮助的AI助手。"},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1
    }
    
    response = requests.post(
        os.environ['DEEPSEEK_API_URL'],
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"
        },
        json=data
    )
    
    if response.status_code == 200:
        response_data = response.json()
        return response_data['choices'][0]['message']['content']
    else:
        return f"错误: {response.status_code} - {response.text}"

# 2. 创建检索器
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 手动实现 RAG 流程
def ask_question(question: str):
    """手动实现 RAG 问答"""
    
    # 1. 检索相关文档
    relevant_docs = retriever.get_relevant_documents(question)
    
    # 2. 构建提示词
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    prompt = f"""基于以下上下文信息，请回答问题。如果你不知道答案，就说不知道。

上下文：
{context}

问题：{question}

请根据上下文提供准确的回答："""
    
    # 3. 调用 DeepSeek
    return call_deepseek(prompt), relevant_docs

print("✅ RAG 系统构建完成！")



#### 见证奇迹的时刻！

现在，让我们向我们的 RAG 应用提问吧！还记得刚才那个标准大模型的模糊回答吗？让我们看看装配了知识库的 RAG 应用表现如何。

刚才的问题：`鲁米纳星球面临什么危险？`

尝试问一些文档中 **没有** 的信息，比如`鲁米纳星球有什么植物？`，观察 AI 会如何回答。它会“诚实”地说不知道，还是会产生幻觉？

In [ ]:
# --- 测试我们的 RAG 系统 ---
print("\n🤖 现在你可以向鲁米纳专家提问了！")
print("输入 '退出' 或 'quit' 来结束对话\n")

while True:
    user_question = input("你的问题: ")
    
    if user_question.lower() in ['退出', 'quit', 'exit']:
        print("再见！👋")
        break
        
    if not user_question.strip():
        print("问题不能为空，请重新输入。")
        continue
    
    print("\n正在思考中...💭")
    try:
        answer, source_docs = ask_question(user_question)
        
        print(f"\n📝 答案: {answer}")
        print("\n📚 参考来源:")
        for i, doc in enumerate(source_docs, 1):
            print(f"{i}. {doc.page_content[:100]}...")
            
    except Exception as e:
        print(f"❌ 出错了: {e}")
    
    print("\n" + "="*50 + "\n")

## 4. 总结与展望

**恭喜你！** 你已经成功地从零到一，构建并运行了一个完整的、交互式的 RAG 应用！

### 4.1 课程回顾与知识梳理

让我们一起快速回顾一下整个旅程：

1.  **理解痛点:** 我们认识到标准大模型存在 **知识盲区** 和 **模型幻觉** 的问题。
2.  **学习 RAG:** 我们学习了 RAG（检索增强生成）如何通过 **“开卷考试”** 的方式解决这些痛点。
3.  **准备工具:** 我们安装了必要的库，并获取了 DeepSeek 的 **API Key**。
4.  **动手实践:**
    *   **加载文档 (Load):** 使用 `TextLoader` 加载了我们的知识库。
    *   **文本切分 (Split):** 使用 `RecursiveCharacterTextSplitter` 将文档切分成小块，并通过交互式组件理解了其原理。
    *   **向量化 (Embed):** 使用 `HuggingFaceEmbeddings` 将文本块转换为向量。
    *   **构建索引 (Store):** 使用 `FAISS` 构建了一个高效的向量数据库。
    *   **整合问答 (Query):** 使用 `RetrievalQA` 链，整合 DeepSeek LLM 和检索器，最终实现了智能问答。

### 4.2 挑战与拓展

我们构建的应用只是一个起点。在真实的、更复杂的场景中，你可能会遇到新的挑战，也有很多可以优化的方向：

*   **提升检索质量:**
    *   尝试不同的 `TextSplitter` 参数或方法。
    *   更换更强大的 `Embedding` 模型。
    *   使用更先进的检索策略，例如 HyDE（假设性文档嵌入）或多查询检索器。
*   **扩展知识库:**
    *   尝试加载不同格式的文档，如 PDF (`PyPDFLoader`)、Word (`UnstructuredWordDocumentLoader`) 甚至整个网站 (`WebBaseLoader`)。
*   **优化交互体验:**
    *   为你的问答机器人增加聊天历史记录，让对话更自然 (`ConversationalRetrievalChain`)。
    *   为它创建一个简单的 Web 界面（例如使用 Streamlit 或 Gradio）。

### 4.3 下一步学习路径

如果你对这个领域意犹未尽，这里有一些精选的资源可以帮助你继续深入探索：

*   **[DeepSeek 官方文档](https://platform.deepseek.com/docs):** 了解更多关于 DeepSeek 模型的功能和使用方法。
*   **[LangChain Python 官方文档](https://python.langchain.com/docs/get_started/introduction):** LangChain 的宝库，包含了海量的教程和 API 文档。
*   **[Sentence-Transformers 库文档](https://www.sbert.net/):** 探索更多强大的开源 Embedding 模型。

**AI 应用开发的世界广阔而精彩，今天的课程只是一个开始。保持好奇，不断学习，你将能创造出更多有价值、有创意的应用！**

### 4.3 下一步学习路径

如果你对这个领域意犹未尽，这里有一些精选的资源可以帮助你继续深入探索：

*   **[DeepSeek 官方文档](https://platform.deepseek.com/docs):** 了解更多关于 DeepSeek 模型的功能和使用方法。
*   **[LangChain Python 官方文档](https://python.langchain.com/docs/get_started/introduction):** LangChain 的宝库，包含了海量的教程和 API 文档。
*   **[Sentence-Transformers 库文档](https://www.sbert.net/):** 探索更多强大的开源 Embedding 模型。

**AI 应用开发的世界广阔而精彩，今天的课程只是一个开始。保持好奇，不断学习，你将能创造出更多有价值、有创意的应用！**